In [2]:
# @title
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!


In [3]:
# @title
!pip install import_ipynb
import import_ipynb
print("import_ipynb installed and imported.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.3 MB/s eta 0:00:00
import_ipynb installed and imported.


In [4]:
# @title
import io
import json
import os
import sys
import types

def load_and_execute_ipynb(file_path):
    module_name = os.path.basename(file_path).replace('.ipynb', '')

    with open(file_path, 'r') as f:
        notebook_content = json.load(f)

    # Create a new module object for the notebook
    # This module will contain all the globals defined in the notebook
    module = types.ModuleType(module_name)
    module.__file__ = file_path
    sys.modules[module_name] = module # Register it in sys.modules

    # Prepare the execution context for the notebook
    exec_globals = module.__dict__
    # Update module's globals with current globals for shared objects (like numpy)
    exec_globals.update(globals())

    # Execute code cells
    for cell in notebook_content['cells']:
        if cell['cell_type'] == 'code':
            python_code_lines = []
            for line in cell['source']:
                if line.strip().startswith('!'):
                    os.system(line.strip()[1:])
                else:
                    python_code_lines.append(line)

            if python_code_lines:
                # Execute the code within the context of the new module's dictionary
                exec(''.join(python_code_lines), exec_globals)

    # Update the main global scope with the content of the notebook's module
    # This makes functions like build_pauli_hamiltonian directly accessible
    globals().update(module.__dict__)

# Manually load and execute necessary ipynb files in correct dependency order
print("Loading and executing hamiltonian.ipynb...")
load_and_execute_ipynb('/content/drive/MyDrive/Colab Notebooks/hamiltonian.ipynb')
print("hamiltonian.ipynb executed successfully.")

print("Loading and executing simulator.ipynb...") # Load simulator first
load_and_execute_ipynb('/content/drive/MyDrive/Colab Notebooks/simulator.ipynb')
print("simulator.ipynb executed successfully.")

print("Loading and executing methods.ipynb...") # Load methods before experiments
load_and_execute_ipynb('/content/drive/MyDrive/Colab Notebooks/methods.ipynb')
print("methods.ipynb executed successfully.")

print("Loading and executing experiments.ipynb...") # Then experiments
load_and_execute_ipynb('/content/drive/MyDrive/Colab Notebooks/experiments.ipynb')
print("experiments.ipynb executed successfully.")

Loading and executing hamiltonian.ipynb...
hamiltonian.ipynb executed successfully.
Loading and executing simulator.ipynb...
simulator.ipynb executed successfully.
Loading and executing methods.ipynb...
methods.ipynb executed successfully.
Loading and executing experiments.ipynb...
experiments.ipynb executed successfully.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from hamiltonian import verify_gradient, build_pauli_hamiltonian
from experiments import scenario_snr, scenario_N, scenario_speed, scenario_K
from simulator import DEFAULT_PARAMS

# ── 1. Gradient check (unchanged) ────────────────────────────────────────────
print("=" * 60)
print("Toy 2-element Hamiltonian check")
print("=" * 60)

a, b, c = 1.2, 0.9, 0.5
H_op = build_pauli_hamiltonian(a, b, c)
print("\nPauli decomposition:")
for term in H_op:
    print(f"  {float(np.real(term.coeffs[0])):+.6f} * {str(term.paulis[0])}")

print()
ok = verify_gradient(gamma=0.4, beta=0.7, a=a, b=b, c=c)
if not ok:
    raise RuntimeError("Gradient check failed.")

# ── 2. Scenarios ──────────────────────────────────────────────────────────────
N_TRIALS = 20

print("\nScenario 1: SNR sweep (dynamic cars)")
df_snr = scenario_snr(snr_db_range=range(0, 31, 5), n_trials=N_TRIALS)
df_snr.to_csv('results_snr.csv', index=False)

print("\nScenario 2: N sweep (dynamic cars)")
df_N = scenario_N(N_values=(16, 32, 64), n_trials=N_TRIALS)
df_N.to_csv('results_N.csv', index=False)

print("\nScenario 3: Speed sweep (NEW)")
df_spd = scenario_speed(speed_values=(5, 10, 20, 30), n_trials=N_TRIALS)
df_spd.to_csv('results_speed.csv', index=False)

print("\nScenario 4: K (number of cars) sweep")
df_K = scenario_K(K_values=((2,2),(4,4),(8,8)), n_trials=N_TRIALS)
df_K.to_csv('results_K.csv', index=False)

# ── 3. Tables ─────────────────────────────────────────────────────────────────
def print_table(df, sweep_col, title):
    print(f"\n{'='*60}\n{title}\n{'='*60}")
    for val in df[sweep_col].unique():
        sub = df[df[sweep_col] == val]
        for m in ['classical','qaoa','hybrid']:
            row = sub[sub.method == m].iloc[0]
            print(f"  {val}  {m:<12}  "
                  f"SR={row['sum_rate_mean']:.3f}  "
                  f"E={row['energy_norm_mean']:.1f}  "
                  f"t={row['time_s_mean']:.3f}s")

print_table(df_snr, 'snr_db',   "Table 1 – SNR sweep")
print_table(df_N,   'N',        "Table 2 – N sweep")
print_table(df_spd, 'speed_ms', "Table 3 – Speed sweep (NEW)")
print_table(df_K,   'K',        "Table 4 – K sweep")

# ── 4. Plots ──────────────────────────────────────────────────────────────────
COLORS  = {'classical':'#185FA5', 'qaoa':'#993C1D', 'hybrid':'#0F6E56'}
MARKERS = {'classical':'o',       'qaoa':'s',        'hybrid':'^'}
LS      = {'classical':'-',       'qaoa':'--',       'hybrid':'-.'}

def _plot(df, xcol, ycol, xlabel, ylabel, title, fname):
    fig, ax = plt.subplots(figsize=(6,4))
    for m in ['classical','qaoa','hybrid']:
        s = df[df.method==m].sort_values(xcol)
        ax.plot(s[xcol], s[ycol+'_mean'],
                color=COLORS[m], marker=MARKERS[m],
                linestyle=LS[m], label=m.capitalize(), linewidth=1.5)
        ax.fill_between(s[xcol],
                        s[ycol+'_mean'] - s[ycol+'_std'],
                        s[ycol+'_mean'] + s[ycol+'_std'],
                        color=COLORS[m], alpha=0.12)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend(); ax.grid(True, linewidth=0.4)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.close()
    print(f"  Saved {fname}")

print("\nGenerating figures...")
_plot(df_snr, 'snr_db',   'sum_rate', 'SNR (dB)',
      'Avg sum-rate (bits/s/Hz)', 'Sum-rate vs SNR — moving cars',
      'fig_sumrate_snr.png')

_plot(df_spd, 'speed_ms', 'sum_rate', 'Car speed (m/s)',
      'Avg sum-rate (bits/s/Hz)', 'Sum-rate vs speed',
      'fig_sumrate_speed.png')

_plot(df_N,   'N',        'sum_rate', 'STAR-RIS elements N',
      'Avg sum-rate (bits/s/Hz)', 'Sum-rate vs N — moving cars',
      'fig_sumrate_N.png')

_plot(df_K,   'K',        'sum_rate', 'Number of cars K',
      'Avg sum-rate (bits/s/Hz)', 'Sum-rate vs K — moving cars',
      'fig_sumrate_K.png')

for df, col, fname in [
    (df_snr, 'snr_db',   'fig_ee_snr.png'),
    (df_spd, 'speed_ms', 'fig_ee_speed.png'),
    (df_N,   'N',        'fig_ee_N.png'),
]:
    df['ee_mean'] = df['sum_rate_mean'] / (df['energy_norm_mean'] + 1e-9)
    df['ee_std']  = df['ee_mean'] * 0.1
    _plot(df, col, 'ee', df[col].name,
          'Energy efficiency', f'Energy efficiency vs {col}', fname)

print("\nDone.")

Toy 2-element Hamiltonian check

Pauli decomposition:
  -1.175000 * II
  -0.475000 * IZ
  -0.325000 * ZI
  -0.125000 * ZZ

  Gradient check:
    d/d_gamma  analytic=0.55765462   FD=0.55765462   err=7.68e-12
    d/d_beta   analytic=0.03530236   FD=0.03530236   err=1.09e-11
  Result: PASS

Scenario 1: SNR sweep (dynamic cars)
  SNR=0 dB....................
  SNR=5 dB....................
  SNR=10 dB....................
  SNR=15 dB....................
  SNR=20 dB....................
  SNR=25 dB....................
  SNR=30 dB....................

Scenario 2: N sweep (dynamic cars)
  N=16....................
  N=32....................
  N=64....................

Scenario 3: Speed sweep (NEW)
  speed=5 m/s....................
  speed=10 m/s....................
  speed=20 m/s................

In [ ]:
import os
from google.colab import files

# Create a zip archive of the 'outputs' folder
zip_filename = 'outputs.zip'
!zip -r "{zip_filename}" "outputs" # Adjust 'outputs' if your folder is nested, e.g., './outputs'

# Download the zip file
files.download(zip_filename)
print(f"'{zip_filename}' should now be downloaded to your local machine.")

  adding: outputs/ (stored 0%)
  adding: outputs/results_snr.csv (deflated 57%)
  adding: outputs/results_N.csv (deflated 54%)
  adding: outputs/results_speed.csv (deflated 55%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'outputs.zip' should now be downloaded to your local machine.
